In [ ]:
import pandas as pd
import numpy as np
from cs_analysis_functions import stats_analysis
from cs_analysis_functions import player_distribution_creater
from multiprocessing import Pool
import os

In [ ]:
data_path = "~/Desktop/CS Data/hltv_match_data_final.csv"
raw_data = pd.read_csv(data_path)
raw_data = raw_data[['match_id', 'game_id', 'match_url', 'date', 'team', 'player', 'map', 'kd_diff', 
                     'assists', 'kast_pct', 'Round_Win_%', 'W/L', 'Rank', 'Points']]

player_ids = raw_data['player'].to_list()
player_ids = list(dict.fromkeys(player_ids))
team_ids = raw_data['team'].to_list()
team_ids = list(dict.fromkeys(team_ids))
map_ids = raw_data['map'].to_list()
map_ids = list(dict.fromkeys(map_ids))

In [ ]:
n_splits = os.cpu_count() - 2
player_chunks = np.array_split(player_ids, n_splits)

if __name__ == "__main__":
    with Pool(processes= n_splits) as pool:
        results = pool.starmap(stats_analysis, [(raw_data, chunk, map_ids) for chunk in player_chunks])
    final_df = pd.concat(results, ignore_index= True)

In [ ]:
final_df = final_df.dropna()
if __name__ == "__main__":
    with Pool(processes= n_splits) as pool:
        results = pool.starmap(player_distribution_creater, [(final_df, chunk, map_ids) for chunk in player_chunks])
    final_dists = pd.concat(results, ignore_index= True)

final_dists.to_parquet('~/Desktop/CS Data/player_distributions.parquet')